<a href="https://colab.research.google.com/github/ShaikSameera56/Nutrition-Tracker/blob/main/Nutrition_Tracker_hackathon_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00


In [2]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.6 MB/s eta 0:00:00


In [3]:
import os, io, re, math, time
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss
from PIL import Image
import gradio as gr
from groq import Groq # Add this import

# --- Helper: load dataset (tries /content path, then upload in Colab) ---
csv_path = '/content/cleaned_normalized_nutrition_dataset.csv'
if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("Upload your dataset CSV now (or cancel to use a small demo).")
        uploaded = files.upload()
        if len(uploaded) > 0:
            csv_path = list(uploaded.keys())[0]
    except Exception:
        pass

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded {csv_path} -> {len(df)} rows")
else:
    # Minimal demo dataset (if user doesn't upload a CSV)
    print("No dataset found. Creating a small demo nutrition dataset.")
    df = pd.DataFrame([
        {'item':'plain rice','calories_per_100g':130,'protein_per_100g':2.7,'carbs_per_100g':28,'fat_per_100g':0.3},
        {'item':'biryani','calories_per_100g':210,'protein_per_100g':5,'carbs_per_100g':25,'fat_per_100g':9},
        {'item':'ice cream','calories_per_100g':207,'protein_per_100g':3.5,'carbs_per_100g':24,'fat_per_100g':11},
        {'item':'samosa','calories_per_100g':300,'protein_per_100g':6,'carbs_per_100g':30,'fat_per_100g':18},
        {'item':'dal','calories_per_100g':120,'protein_per_100g':8,'carbs_per_100g':15,'fat_per_100g':2},
        {'item':'paneer','calories_per_100g':265,'protein_per_100g':18,'carbs_per_100g':1.2,'fat_per_100g':20},
    ])
    df.to_csv('/content/demo_nutrition.csv', index=False)
    csv_path = '/content/demo_nutrition.csv'

# --- Normalize columns & friendly name detection ---
def find_col(cands):
    for c in df.columns:
        lc = c.lower()
        for cand in cands:
            if cand in lc:
                return c
    return None

item_col = find_col(['item','food','name']) or df.columns[0]
cal_col  = find_col(['calor','kcal']) or 'calories_per_100g'
prot_col = find_col(['protein']) or 'protein_per_100g'
carb_col = find_col(['carb']) or 'carbs_per_100g'
fat_col  = find_col(['fat']) or 'fat_per_100g'

# Ensure lowercase item names for matching
df[item_col] = df[item_col].astype(str).str.lower().str.strip()
df.fillna(0, inplace=True)

# Ensure numeric columns exist and convert to floats (create zeros if missing)
for col in [cal_col, prot_col, carb_col, fat_col]:
    if col not in df.columns:
        df[col] = 0.0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Build a map and lists used by FAISS
items = df[item_col].tolist()
food_map = df.to_dict(orient='records')

# --- Groq API setup ---
from google.colab import userdata
try:
    groq_api_key = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=groq_api_key)
    print("Groq client initialized.")
except:
    print("Groq API key not found. Please set 'GROQ_API_KEY' in Colab secrets.")
    client = None

Loaded /content/cleaned_normalized_nutrition_dataset.csv -> 1068 rows
Groq client initialized.


In [4]:
!huggingface-cli login


Hint: A new version of huggingface_hub (2.2.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [6]:
import os
from huggingface_hub import get_token

# Load the token using the modern huggingface_hub helper function
token = get_token()

if token:
    # Set the token as an environment variable
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    print("Hugging Face token loaded from file and set as environment variable.")
else:
    print("No Hugging Face token found. Please ensure you have logged in correctly.")

# --- Sentence Transformer + FAISS index ---
print("Loading sentence-transformers model (all-MiniLM-L6-v2)...")
# Explicitly pass the token if available
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN') # Get token from Colab secrets
except Exception:
    hf_token = None

token_to_use = hf_token if hf_token else token
model = SentenceTransformer('all-MiniLM-L6-v2', use_auth_token=token_to_use if token_to_use else None)

embeddings = model.encode(items, convert_to_numpy=True, show_progress_bar=True)
embeddings = np.ascontiguousarray(embeddings.astype('float32'))

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index built, dim={dim}, {index.ntotal} vectors")

Hugging Face token loaded from file and set as environment variable.
Loading sentence-transformers model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.13/dist-packages/sentence_transformers/sentence_transformer/model.py:178: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in a future release of SentenceTransformers.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

FAISS index built, dim=384, 1068 vectors


In [7]:
# --- New Vector Store for User Data ---
print("Creating a dedicated FAISS index for user meal data...")
user_data_embeddings = np.array([], dtype='float32').reshape(0, dim)
user_data_index = faiss.IndexFlatL2(dim)
user_data_map = [] # stores a list of dictionaries for each user meal entry
print(f"User data FAISS index created, dim={dim}")

Creating a dedicated FAISS index for user meal data...
User data FAISS index created, dim=384


In [8]:
# --- Sentence Transformer + FAISS index ---
print("Loading sentence-transformers model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(items, convert_to_numpy=True, show_progress_bar=True)
embeddings = np.ascontiguousarray(embeddings.astype('float32'))

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index built, dim={dim}, {index.ntotal} vectors")

Loading sentence-transformers model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

FAISS index built, dim=384, 1068 vectors


In [9]:
# --- Retrieval function ---
def retrieve_food_info(query, top_k=3):
    qv = model.encode([query], convert_to_numpy=True).astype('float32')
    D, I = index.search(qv, top_k)
    results = []
    for idx in I[0]:
        if idx < 0 or idx >= len(food_map):
            continue
        r = food_map[idx]
        results.append({
            'food_item': r[item_col],
            'calories_per_100g': float(r.get(cal_col,0)),
            'protein_per_100g': float(r.get(prot_col,0)),
            'carbs_per_100g': float(r.get(carb_col,0)),
            'fat_per_100g': float(r.get(fat_col,0))
        })
    return results

# --- Quantity extraction with LLM ---
def analyze_meal_with_llm(user_input):
    if not client:
        return "LLM parsing is disabled. Please set your Groq API key."
    prompt = f"""
    You are a food parser. Extract the food items and their quantities from the following text.
    Your output must be a single JSON list of objects.
    Each object should have "food_item" (string) and "quantity_g" (integer, in grams).
    If a quantity is not specified, assume 100g.
    If a quantity is given in units like "plate", "bowl", "cup", or "slice", convert it to grams using the following mappings:
    - plate: 250g
    - bowl: 200g
    - cup: 150g
    - piece: 50g
    - slice: 30g
    - scoop: 60g
    Example 1: "two plates of rice and one scoop of ice cream"
    Output: {{"items": [{{"food_item": "rice", "quantity_g": 500}}, {{"food_item": "ice cream", "quantity_g": 60}}]}}
    Example 2: "150g biryani, dal"
    Output: {{"items": [{{"food_item": "biryani", "quantity_g": 150}}, {{"food_item": "dal", "quantity_g": 100}}]}}
    Text to parse: "{user_input}"
    """
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama3-8b-8192", # or another Groq model
            response_format={"type": "json_object"}
        )
        response_json = chat_completion.choices[0].message.content
        import json
        return json.loads(response_json)
    except Exception as e:
        print(f"Error calling Groq API: {e}")
        return None

# --- Main analyzer (updated to use LLM) ---
def analyze_meal_text(user_input):
    llm_output = analyze_meal_with_llm(user_input)
    if not llm_output:
        # Fallback to the old heuristic method if LLM fails
        return analyze_meal_text_fallback(user_input)

    breakdown = []
    totals = {'calories':0.0,'protein':0.0,'carbs':0.0,'fat':0.0}

    for item in llm_output.get('items', []):
        food_text = item.get('food_item')
        grams = item.get('quantity_g', 100)

        results = retrieve_food_info(food_text, top_k=1)
        if len(results) == 0:
            matched = {'food_item': food_text, 'calories_per_100g': 0, 'protein_per_100g': 0, 'carbs_per_100g': 0, 'fat_per_100g': 0}
        else:
            matched = results[0]

        factor = grams / 100.0
        cal = matched['calories_per_100g'] * factor
        prot = matched['protein_per_100g'] * factor
        carbs = matched['carbs_per_100g'] * factor
        fat = matched['fat_per_100g'] * factor

        breakdown.append({
            'food_item': matched['food_item'],
            'grams': round(grams, 1),
            'calories': round(cal, 1),
            'protein': round(prot, 1),
            'carbs': round(carbs, 1),
            'fat': round(fat, 1)
        })
        totals['calories'] += cal
        totals['protein'] += prot
        totals['carbs'] += carbs
        totals['fat'] += fat

    for k in totals:
        totals[k] = round(totals[k], 1)

    return breakdown, totals

# Retain the old function as a fallback
def analyze_meal_text_fallback(user_input):
    segs = re.split(r',|;|\band\b|&', user_input, flags=re.IGNORECASE)
    breakdown = []
    totals = {'calories':0.0,'protein':0.0,'carbs':0.0,'fat':0.0}

    number_map = {
        'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,'eight':8,'nine':9,'ten':10,
        'a':1,'an':1,'half':0.5
    }
    unit_map = {
        'plate':250,'plates':250,'bowl':200,'bowls':200,'cup':150,'cups':150,'scoop':60,'scoops':60,
        'piece':50,'pieces':50,'slice':30,'slices':30,'tbsp':15,'tablespoon':15,'tsp':5,'teaspoon':5,
        'g':1,'gram':1,'grams':1,'kg':1000,'ml':1
    }
    default_portion_g = 100

    def extract_quantity_single(seg):
        seg = seg.lower()
        m = re.search(r'(\d+(?:\.\d+)?)\s*(g|gram|grams)\b', seg)
        if m: return float(m.group(1))
        m = re.search(r'(\d+)\s*/\s*(\d+)', seg)
        if m:
            try: return float(m.group(1))/float(m.group(2)) * default_portion_g
            except: pass
        num_pat = r'(\d+(?:\.\d+)?)'
        word_num_pat = r'\b(' + '|'.join(re.escape(k) for k in number_map.keys()) + r')\b'
        unit_pat = r'(' + '|'.join(re.escape(k) for k in unit_map.keys()) + r')\b'
        m = re.search(num_pat + r'\s*' + unit_pat, seg)
        if m:
            num = float(m.group(1))
            unit = m.group(2)
            return num * unit_map.get(unit, default_portion_g)
        m = re.search(word_num_pat + r'\s*' + unit_pat, seg)
        if m:
            num = number_map.get(m.group(1), 1)
            unit = m.group(2)
            return num * unit_map.get(unit, default_portion_g)
        m = re.search(num_pat, seg)
        if m:
            val = float(m.group(1))
            if val <= 10: return val * default_portion_g
            else: return val
        m = re.search(word_num_pat, seg)
        if m:
            val = number_map.get(m.group(1),1)
            return val * default_portion_g
        return None

    def clean_food_text(seg):
        seg = seg.lower()
        seg = re.sub(r'(\d+(?:\.\d+)?\s*(g|gram|grams|kg|ml)|\d+\s*/\s*\d+)', '', seg)
        seg = re.sub(r'\b(' + '|'.join(re.escape(k) for k in list(number_map.keys()) + list(unit_map.keys())) + r')\b', '', seg)
        seg = re.sub(r'\b(of|with|and|&)\b', ' ', seg)
        seg = re.sub(r'[^a-z0-9 ]+', ' ', seg)
        seg = ' '.join(seg.split()).strip()
        return seg

    for seg in segs:
        seg = seg.strip()
        if not seg: continue
        grams = extract_quantity_single(seg)
        food_text = clean_food_text(seg)
        if grams is None: grams = default_portion_g
        if not food_text: food_text = seg

        results = retrieve_food_info(food_text, top_k=1)
        if len(results) == 0:
            matched = {'food_item': food_text, 'calories_per_100g': 0, 'protein_per_100g': 0, 'carbs_per_100g': 0, 'fat_per_100g': 0}
        else:
            matched = results[0]

        factor = grams / 100.0
        cal = matched['calories_per_100g'] * factor
        prot = matched['protein_per_100g'] * factor
        carbs = matched['carbs_per_100g'] * factor
        fat = matched['fat_per_100g'] * factor

        breakdown.append({
            'food_item': matched['food_item'],
            'grams': round(grams,1),
            'calories': round(cal,1),
            'protein': round(prot,1),
            'carbs': round(carbs,1),
            'fat': round(fat,1)
        })
        totals['calories'] += cal
        totals['protein'] += prot
        totals['carbs'] += carbs
        totals['fat'] += fat

    for k in totals:
        totals[k] = round(totals[k],1)
    return breakdown, totals

In [10]:
# --- Image helper for plotting -> PIL image for Gradio ---
def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).convert('RGB')
    plt.close(fig)
    return img

def make_nutrition_pie(totals, title='Nutrition'):
    fig, ax = plt.subplots(figsize=(4,4))
    labels = []
    values = []
    if totals['protein']>0:
        labels.append('Protein'); values.append(totals['protein'])
    if totals['carbs']>0:
        labels.append('Carbs'); values.append(totals['carbs'])
    if totals['fat']>0:
        labels.append('Fat'); values.append(totals['fat'])
    if len(values)==0:
        labels=['None']; values=[1]
    ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(title)
    return fig_to_pil(fig)

# --- In-memory users store (simple) ---
USERS = {}  # username -> {password, height, weight, age, health_conditions, meals:[]}
# meal record: {'timestamp': iso, 'input': text, 'breakdown': [...], 'totals': {...}}

# --- Gradio callbacks (signup/login/analyze/weekly) ---
def signup_action(username, password, height, weight, age, health):
    if not username or not password:
        return "Please provide username & password."
    if username in USERS:
        return f"Username '{username}' already exists."
    USERS[username] = {
        'password': password,
        'height': height or None,
        'weight': weight or None,
        'age': age or None,
        'health_conditions': health or '',
        'meals': []
    }
    return f"✅ Signed up '{username}'. Now log in."

def login_action(username, password, state):
    if username not in USERS:
        return "No such user. Please signup first.", None
    if USERS[username]['password'] != password:
        return "Incorrect password.", None
    # state holds current username
    return f"✅ Logged in as {username}", username

def analyze_action(meal_text, current_username):
    if not current_username:
        return "Please login first.", None, None
    if not meal_text or meal_text.strip()=="":
        return "Please type what you ate.", None, None
    breakdown, totals = analyze_meal_text(meal_text)
    # log meal
    USERS[current_username]['meals'].append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'input': meal_text,
        'breakdown': breakdown,
        'totals': totals
    })
    # Prepare textual report
    lines = ["🍽 Meal breakdown:"]
    for b in breakdown:
        lines.append(f"- {b['food_item'].title()} — {b['grams']} g — {b['calories']} kcal, {b['protein']}g protein, {b['carbs']}g carbs, {b['fat']}g fat")
    lines.append("")
    lines.append("🔥 Totals:")
    lines.append(f"Calories: {totals['calories']:.1f} kcal")
    lines.append(f"Protein: {totals['protein']:.1f} g")
    lines.append(f"Carbs: {totals['carbs']:.1f} g")
    lines.append(f"Fat: {totals['fat']:.1f} g")
    # simple recommendations
    lines.append("")
    lines.append("💡 Suggestions:")
    if totals['protein'] < 40:
        lines.append("- Protein looks low: add legumes, paneer, yogurt, or eggs (if not vegetarian restriction).")
    if totals['carbs'] > 300:
        lines.append("- Carbs are on the high side: reduce rice/bread/sugary items.")
    if totals['fat'] > 70:
        lines.append("- Fat is high: avoid fried/oily foods and reduce portions.")
    text_report = "\n".join(lines)
    img = make_nutrition_pie(totals, title='Meal Nutrition')
    return text_report, img, totals

def weekly_report_action(current_username):
    if not current_username:
        return "Please login first.", None
    records = USERS[current_username]['meals']
    if not records:
        return "No meals logged yet.", None
    one_week_ago = datetime.now() - timedelta(days=7)
    week_records = [r for r in records if datetime.strptime(r['timestamp'],'%Y-%m-%d %H:%M:%S') > one_week_ago]
    if not week_records:
        return "No meals in the last 7 days.", None
    tot_cal = tot_prot = tot_carbs = tot_fat = 0.0
    for r in week_records:
        tot_cal += r['totals'].get('calories',0)
        tot_prot += r['totals'].get('protein',0)
        tot_carbs += r['totals'].get('carbs',0)
        tot_fat += r['totals'].get('fat',0)
    totals = {'calories':round(tot_cal,1),'protein':round(tot_prot,1),'carbs':round(tot_carbs,1),'fat':round(tot_fat,1)}
    lines = [f"Weekly summary ({len(week_records)} meals):",
             f"Total calories: {totals['calories']:.1f} kcal",
             f"Protein: {totals['protein']:.1f} g",
             f"Carbs: {totals['carbs']:.1f} g",
             f"Fat: {totals['fat']:.1f} g"]
    # quick flagging
    lines.append("")
    if totals['protein'] < 300:
        lines.append("- Weekly protein is relatively low; consider adding protein-rich meals across the week.")
    if totals['carbs'] > 2000:
        lines.append("- Weekly carbs seem high; reduce high-carb meals and prefer whole grains.")
    if totals['fat'] > 700:
        lines.append("- Weekly fats are high; try leaner cooking, fewer fried items.")
    text = "\n".join(lines)
    img = make_nutrition_pie(totals, title='Weekly Nutrition')
    return text, img

# --- Build Gradio UI ---
with gr.Blocks() as demo:
    gr.Markdown("## 🍽️ Meal Tracker — Signup, Parse, FAISS retrieval, Charts")
    with gr.Row():
        username = gr.Textbox(label="Username", placeholder="choose a username")
        password = gr.Textbox(label="Password", placeholder="password", type="password")
        signup_btn = gr.Button("Signup", variant="primary")
        login_btn = gr.Button("Login", variant="secondary")
        login_status = gr.Textbox(label="Status", interactive=False)
    # hidden state to hold current logged-in username
    current_user_state = gr.State(value=None)

    signup_btn.click(fn=signup_action, inputs=[username,password, gr.Textbox(visible=False,value=""), gr.Textbox(visible=False,value=""), gr.Textbox(visible=False,value=""), gr.Textbox(visible=False,value="")], outputs=[login_status])
    # Note: signup_action signature expects more args; we passed dummy textboxes for height/weight/age/health to keep UI simple.
    def login_wrapper(u,p,state):
        msg, user = login_action(u,p,state)
        return msg, user
    login_btn.click(fn=login_wrapper, inputs=[username,password,current_user_state], outputs=[login_status,current_user_state])

    gr.Markdown("### Analyze a Meal")
    meal_input = gr.Textbox(label="Describe meal (e.g. '2 plates biryani and 1 scoop ice cream')", lines=2)
    analyze_btn = gr.Button("Analyze Meal", variant="primary")
    meal_text_output = gr.Textbox(label="Report", lines=10, interactive=False)
    meal_pie = gr.Image(type="pil", label="Nutrition Pie Chart")
    analyze_btn.click(fn=lambda m,u: ("Please login first.", None, None) if not u else analyze_action(m,u),
                      inputs=[meal_input,current_user_state],
                      outputs=[meal_text_output, meal_pie, gr.Textbox(visible=False)])

    gr.Markdown("### Weekly Report (last 7 days)")
    weekly_btn = gr.Button("Generate Weekly Report", variant="secondary")
    weekly_text = gr.Textbox(label="Weekly Report", lines=8, interactive=False)
    weekly_img = gr.Image(type="pil", label="Weekly Nutrition Pie")
    weekly_btn.click(fn=lambda u: ("Please login first.", None) if not u else weekly_report_action(u),
                      inputs=[current_user_state], outputs=[weekly_text, weekly_img])

    gr.Markdown("----\nNotes: This runs fully in this Colab session (in-memory users). For production you'd save users to a DB, secure passwords, and optionally use an LLM API (Groq/OpenAI) to parse meal text more accurately.")

# Launch Gradio app
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d678f30bfa98783075.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
